#### PURPOSE:
##### This script downloads raw datasets required for the ML project:
+ 1. TTC Bus Delay data
+ 2. Toronto Weather data
+ 3. TTC GTFS (routes + schedules)
+ 4. Road restriction data

#### OUTPUT:
##### All datasets are saved as CSV files inside /data folder

In [30]:
pip install requests

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import shutil
import requests
import zipfile
import pandas as pd
from pathlib import Path
from io import StringIO

In [2]:
# CONFIG
# Base API for Toronto open datasets
BASE_URL = "https://ckan0.cf.opendata.inter.prod-toronto.ca"

# Folder structure for raw datasets
DATA_DIR = Path("data")
TTC_DIR = DATA_DIR/"ttc_bus"
WEATHER_DIR = DATA_DIR/"weather"
GTFS_DIR = DATA_DIR/"ttc_gtfs"
ROAD_DIR = DATA_DIR/"road_restrictions"

target_years = ("2022", "2023", "2024", "2025")

# Create folder helper
def make_folder(path):
    os.makedirs(path, exist_ok=True)


# Download file helper
def download_file(url, path):
    r = requests.get(url, stream=True)
    r.raise_for_status()

    with open(path, "wb") as f:
        for chunk in r.iter_content(8192):
            f.write(chunk)

#### TTC BUS DELAYS

In [48]:
url = f"{BASE_URL}/api/3/action/package_show"
params = {"id": "ttc-bus-delay-data"}

package = requests.get(url, params=params)
package.raise_for_status()
package = package.json()

TTC_DIR.mkdir(parents=True, exist_ok=True)

for resource in package["result"]["resources"]:

    resource_name = resource.get("name", "")
    resource_url = resource.get("url", "")

    # check both name + url)
    if not any(year in resource_name or year in resource_url for year in target_years):
        continue

    if not resource_url:
        continue

    # keep only real data files
    if not resource_url.lower().endswith((".csv", ".xlsx")):
        continue

    filename = resource_url.split("/")[-1]
    file_path = data_dir / filename

    # SKIP if already exists (important for local runs)
    if file_path.exists():
        print(f"Skipping (already exists): {filename}")
        continue

    print(f"Downloading: {filename}")
    download_file(resource_url, file_path)

print("Download complete")

Downloading: ttc-bus-delay-data-2022.xlsx
Downloading: ttc-bus-delay-data-2023.xlsx
Downloading: ttc-bus-delay-data-2024.xlsx
Downloading: ttc-bus-delay-data-since-2025.csv
Download complete


#### TTC ROUTES & SCHEDULES DATA

In [5]:
if GTFS_DIR.exists():
    shutil.rmtree(GTFS_DIR)

make_folder(GTFS_DIR)

package = requests.get(
        f"{BASE_URL}/api/3/action/package_show",
        params={"id": "ttc-routes-and-schedules"}
    ).json()

print("Downloading GTFS files...")

for resource in package["result"]["resources"]:
    file_url = resource.get("url", "")

    if not file_url:
        continue

    filename = file_url.split("/")[-1]
    file_path = GTFS_DIR / filename

    print(f"Downloading: {filename}")
    download_file(file_url, file_path)

print("Extracting ZIP files...")

for zip_file in GTFS_DIR.glob("*.zip"):

    with zipfile.ZipFile(zip_file, "r") as z:
        z.extractall(GTFS_DIR)

    os.remove(zip_file)

print("GTFS download complete.")

Downloading: opendata_ttc_schedules.zip
Extracting ZIP files...
GTFS download complete.


#### ROAD RESTRICTIONS DATA

In [9]:
make_folder(ROAD_DIR)
package = requests.get(f"{BASE_URL}/api/3/action/package_show", params={"id": "road-restrictions"}).json()

for resource in package["result"]["resources"]:
    if str(resource.get("format", "")).lower() != "csv":
        continue

    file_url = resource.get("url", "")

    if not file_url:
        continue

    filename = file_url.split("/")[-1]

    if not filename.endswith(".csv"):
        filename = "road_restrictions.csv"

    file_path = ROAD_DIR / filename

    print(f"Downloading: {filename}")
    download_file(file_url, file_path)

print("Road restrictions download complete.")

Downloading: road_restrictions.csv
Road restrictions download complete.


#### WEATHER DATA

In [8]:
# WEATHER CONFIG
STATION_ID = 48549 # Toronto station
START_YEAR = 2022
END_YEAR = 2025
        
make_folder(WEATHER_DIR)
output_file = WEATHER_DIR / "toronto_weather_2022_2025.csv"
    
# Skip if already exists
if output_file.exists():
   print("Weather dataset already exists. Skipping download.")
   exit()
    
# Download data
all_data = []
for year in range(START_YEAR, END_YEAR + 1):
    for month in range(1, 13):
        url = (
                "https://climate.weather.gc.ca/climate_data/bulk_data_e.html"
                f"?format=csv"
                f"&stationID={STATION_ID}"
                f"&Year={year}"
                f"&Month={month}"
                f"&Day=1"
                f"&timeframe=1"
            )

        try:
                text = requests.get(url).text
                header_row = next(
                    (i for i, line in enumerate(text.split("\n"))
                     if "Climate ID" in line),
                    None)

                if header_row is None:
                    continue

                df = pd.read_csv(
                    StringIO(text),
                    skiprows=header_row)

                df["year"] = year
                df["month"] = month
                all_data.append(df)

        except:
                pass

# combine
weather_df = pd.concat(all_data, ignore_index=True)

if "Date/Time" in weather_df.columns:
    weather_df["Date/Time"] = pd.to_datetime(weather_df["Date/Time"], errors="coerce")

# SAVE FILE
weather_df.to_csv(output_file, index=False)
print("Weather data download complete.")

Weather dataset already exists. Skipping download.
Weather data download complete.
